# Cross-Lingual Query Expansion Experiment
This notebook runs an embedding-based query expansion experiment on the COSQA dataset to assist with cross-lingual (Indonesian to English) code retrieval.


## 1. Install Dependencies


In [16]:
!pip install -q sentence-transformers datasets pandas numpy tqdm faiss-cpu

## 2. Setup Google Drive (Optional)
If you have `cosqa_queries_indonesian.csv` on your Google Drive, mount it here. Otherwise, you can upload the file directly to the Colab environment.


In [ ]:
from google.colab import drive
# drive.mount("/content/drive")
# !cp /content/drive/MyDrive/cosqa_queries_indonesian.csv .

## 3. Define Imports


In [17]:
import logging
import json
import pandas as pd
import numpy as np
import torch
from pathlib import Path
from typing import List, Dict, Optional, Tuple
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from tqdm.auto import tqdm
from dataclasses import dataclass

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

## 4. Define Dense Retriever
This class uses Multilingual E5 embeddings to encode queries and documents, and performs dense retrieval.


In [18]:
class DenseRetriever:
    QUERY_PREFIX = "query: "
    PASSAGE_PREFIX = "passage: "
    
    def __init__(
        self,
        model_name: str = "intfloat/multilingual-e5-small",
        device: Optional[str] = None,
        batch_size: int = 32,
        normalize_embeddings: bool = True,
    ):
        self.model_name = model_name
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.batch_size = batch_size
        self.normalize_embeddings = normalize_embeddings
        
        logger.info(f"Loading retriever model: {model_name} on {self.device}")
        self.model = SentenceTransformer(model_name)
        self.model.to(self.device)
        self.corpus_embeddings = None
        self.corpus_ids = None

    def encode_queries(self, queries: List[str]) -> np.ndarray:
        prefixed_queries = [self.QUERY_PREFIX + q for q in queries]
        return self.model.encode(
            prefixed_queries,
            batch_size=self.batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
            device=self.device,
            normalize_embeddings=self.normalize_embeddings,
        )

    def encode_corpus(self, corpus: List[Dict[str, str]]) -> Tuple[np.ndarray, List[str]]:
        texts = [doc.get("text", "") or doc.get("code", "") for doc in corpus]
        doc_ids = [doc.get("id", str(i)) for i, doc in enumerate(corpus)]
        prefixed_texts = [self.PASSAGE_PREFIX + t for t in texts]
        
        embeddings = self.model.encode(
            prefixed_texts,
            batch_size=self.batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
            device=self.device,
            normalize_embeddings=self.normalize_embeddings,
        )
        self.corpus_embeddings = embeddings
        self.corpus_ids = doc_ids
        return embeddings, doc_ids

    def retrieve(self, queries: List[str], corpus: List[Dict[str, str]], top_k: int = 100) -> List[List[Dict]]:
        query_embeddings = self.encode_queries(queries)
        
        if self.corpus_embeddings is None:
            corpus_embeddings, doc_ids = self.encode_corpus(corpus)
        else:
            corpus_embeddings = self.corpus_embeddings
            doc_ids = self.corpus_ids
            
        similarities = np.matmul(query_embeddings, corpus_embeddings.T)
        
        results = []
        for i in range(len(queries)):
            scores = similarities[i]
            top_indices = np.argsort(scores)[-top_k:][::-1]
            result = [
                {"id": doc_ids[idx], "score": float(scores[idx]), "rank": r + 1}
                for r, idx in enumerate(top_indices)
            ]
            results.append(result)
        return results

## 5. Define BM25 Retriever


In [19]:
from collections import Counter

class BM25Retriever:
    def __init__(self, k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b = b
        self.corpus = None
        self.doc_len = None
        self.avgdl = None
        self.doc_freqs = None
        self.idf = None
        self.vocab = None
    
    def fit(self, corpus: List[Dict[str, str]]):
        self.corpus = corpus
        self.doc_len = []
        self.doc_freqs = []
        self.vocab = {}
        
        for doc in corpus:
            text = doc.get("text", "") or doc.get("code", "")
            tokens = text.lower().split()
            self.doc_len.append(len(tokens))
            freqs = {}
            for token in tokens:
                if token not in self.vocab:
                    self.vocab[token] = len(self.vocab)
                freqs[token] = freqs.get(token, 0) + 1
            self.doc_freqs.append(freqs)
            
        self.avgdl = sum(self.doc_len) / len(self.doc_len)
        N = len(corpus)
        self.idf = {}
        df = Counter()
        for freqs in self.doc_freqs:
            for token in freqs:
                df[token] += 1
        
        for token, freq in df.items():
            self.idf[token] = np.log((N - freq + 0.5) / (freq + 0.5) + 1)
            
    def retrieve(self, queries: List[str], top_k: int = 100) -> List[List[Dict]]:
        results = []
        for query in tqdm(queries, desc="BM25 Retrieving"):
            query_tokens = query.lower().split()
            scores = []
            for i in range(len(self.corpus)):
                doc_len = self.doc_len[i]
                freqs = self.doc_freqs[i]
                score = 0.0
                for token in query_tokens:
                    if token not in freqs: continue
                    freq = freqs[token]
                    idf = self.idf.get(token, 0)
                    numerator = freq * (self.k1 + 1)
                    denominator = freq + self.k1 * (1 - self.b + self.b * doc_len / self.avgdl)
                    score += idf * numerator / denominator
                scores.append((i, score))
            scores.sort(key=lambda x: x[1], reverse=True)
            results.append([
                {"id": self.corpus[idx].get("id", str(idx)), "score": float(sc), "rank": r + 1}
                for r, (idx, sc) in enumerate(scores[:top_k])
            ])
        return results

## 6. Define Cross-Lingual Query Expander


In [20]:
@dataclass
class ExpansionResult:
    original_query: str
    expanded_query: str
    expansion_terms: List[str]
    method: str
    metadata: Dict

class CrossLingualEmbeddingExpander:
    def __init__(self, model_name: str = "intfloat/multilingual-e5-small", device: Optional[str] = None):
        self.model_name = model_name
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        logger.info(f"Loading expander model: {model_name} on {self.device}")
        self.model = SentenceTransformer(model_name)
        self.model.to(self.device)
        self._term_vocabulary = self._load_bilingual_vocabulary()
        
    def _load_bilingual_vocabulary(self) -> List[str]:
        indonesian_to_english = {
            "fungsi": "function", "kelas": "class", "metode": "method",
            "variabel": "variable", "parameter": "parameter", "mengembalikan": "return",
            "perulangan": "loop", "kondisi": "condition", "array": "array",
            "daftar": "list", "kamus": "dictionary", "teks": "string",
            "bilangan": "integer", "boolean": "boolean", "baca": "read",
            "tulis": "write", "buka": "open", "tutup": "close", "file": "file",
            "data": "data", "database": "database", "tabel": "table",
            "baris": "row", "kolom": "column", "query": "query",
            "simpan": "save", "ambil": "get", "hapus": "delete",
            "ubah": "update", "tambah": "add", "cari": "search",
            "urut": "sort", "filter": "filter", "error": "error",
            "kesalahan": "error", "peringatan": "warning",
        }
        return list(set(indonesian_to_english.values()))
        
    def _identify_indonesian_terms(self, query: str) -> List[str]:
        indo_terms = ["fungsi", "kelas", "metode", "variabel", "parameter", "loop", "array", "daftar", "kamus", "string", "integer", "boolean", "baca", "tulis", "buka", "tutup", "file", "data", "database", "tabel", "baris", "kolom", "query", "simpan", "ambil", "hapus", "ubah", "tambah", "cari", "urut", "filter", "error", "kesalahan"]
        query_lower = query.lower()
        return [term for term in indo_terms if term in query_lower]
        
    def _translate_term(self, indo_term: str) -> Optional[str]:
        mapping = {"fungsi": "function", "kelas": "class", "metode": "method", "variabel": "variable", "parameter": "parameter", "loop": "loop", "array": "array", "daftar": "list", "kamus": "dictionary", "string": "string", "integer": "integer", "boolean": "boolean", "baca": "read", "tulis": "write", "buka": "open", "tutup": "close", "file": "file", "data": "data", "database": "database", "tabel": "table", "baris": "row", "kolom": "column", "query": "query", "simpan": "save", "ambil": "get", "hapus": "delete", "ubah": "update", "tambah": "add", "cari": "search", "urut": "sort", "filter": "filter", "error": "error", "kesalahan": "error"}
        return mapping.get(indo_term.lower())

    def expand(self, query: str, num_terms: int = 5) -> ExpansionResult:
        indonesian_terms = self._identify_indonesian_terms(query)
        query_embedding = self.model.encode([query], convert_to_numpy=True, device=self.device, show_progress_bar=False)
        term_embeddings = self.model.encode(self._term_vocabulary, convert_to_numpy=True, device=self.device, show_progress_bar=False)
        
        norm1 = query_embedding / np.linalg.norm(query_embedding, axis=1, keepdims=True)
        norm2 = term_embeddings / np.linalg.norm(term_embeddings, axis=1, keepdims=True)
        similarities = np.dot(norm1, norm2.T)[0]
        
        top_k_indices = np.argsort(similarities)[-num_terms:][::-1]
        expansion_terms = [self._term_vocabulary[i] for i in top_k_indices if similarities[i] > 0.1][:num_terms]
        
        for indo_term in indonesian_terms:
            eng_term = self._translate_term(indo_term)
            if eng_term and eng_term not in expansion_terms:
                expansion_terms.insert(0, eng_term)
                
        expanded_query = f"{query} {' '.join(expansion_terms)}"
        
        return ExpansionResult(
            original_query=query, expanded_query=expanded_query,
            expansion_terms=expansion_terms, method="cross_lingual_embedding",
            metadata={"indonesian_terms_found": indonesian_terms, "model": self.model_name}
        )

## 7. Load Data & Evaluation Utils


In [21]:
def load_cosqa_data():
    translations_file = "cosqa_queries_indonesian.csv"
    if Path(translations_file).exists():
        trans_df = pd.read_csv(translations_file, sep="|")
        translations = dict(zip(trans_df['qid'], trans_df['query_id']))
    else:
        logger.warning(f"{translations_file} not found. Using English dataset without translations.")
        translations = {}
        
    logger.info("Loading COSQA from huggingface datasets...")
    queries_corpus_dataset = load_dataset("CoIR-Retrieval/cosqa-queries-corpus")
    qrels_dataset = load_dataset("CoIR-Retrieval/cosqa-qrels")
    
    # Process corpus
    corpus_data = queries_corpus_dataset['corpus']
    corpus = [{"id": str(item["_id"]), "text": item.get("text", "")} for item in corpus_data]
    
    # Process queries & qrels
    query_data = queries_corpus_dataset['queries']
    queries_en = {str(item["_id"]): item.get("text", "") for item in query_data}
    
    qrels_data = qrels_dataset['test']
    qrels = {}
    for item in qrels_data:
        qid = str(item['query_id'])
        doc_id = str(item['corpus_id'])
        score = int(item['score'])
        if qid not in qrels: qrels[qid] = {}
        qrels[qid][doc_id] = score
        
    # Process indonesian queries
    queries_id = {}
    for qid, qtext in queries_en.items():
        queries_id[qid] = translations.get(qid, qtext)
        
    # Subsampling for demonstration (Optional: comment out for full evaluation)
    # SUB_SIZE = 100
    # keys = list(queries_en.keys())[:SUB_SIZE]
    # queries_en = {k: queries_en[k] for k in keys}
    # queries_id = {k: queries_id[k] for k in keys}
    
    return corpus, queries_en, queries_id, qrels

def evaluate_results(results: List[Dict], qrels: Dict[str, Dict[str, int]], top_k: int = 10):
    hits = 0
    total_relevant = 0
    ndcg_sum = 0
    evaluated = 0
    
    for result in results:
        qid = str(result["qid"])
        if qid not in qrels: continue
        relevant_docs = set([str(k) for k, v in qrels[qid].items() if v == 1])
        if not relevant_docs: continue
        total_relevant += len(relevant_docs)
        
        retrieved_ids = [str(doc["id"]) for doc in result["retrieved"][:top_k]]
        hits += len(set(retrieved_ids) & relevant_docs)
        
        dcg = sum(1 / np.log2(i + 2) for i, doc_id in enumerate(retrieved_ids) if doc_id in relevant_docs)
        idcg = sum(1 / np.log2(i + 2) for i in range(min(len(relevant_docs), top_k)))
        if idcg > 0:
            ndcg_sum += dcg / idcg
            evaluated += 1
            
    return {
        "NDCG@10": ndcg_sum / evaluated if evaluated > 0 else 0,
        "Recall@10": hits / total_relevant if total_relevant > 0 else 0,
        "Hits": hits, "Total Relevant": total_relevant, "Evaluated": evaluated
    }
    
def save_detailed_results(results: List[Dict], output_path: str, queries_english: Dict[str, str] = None):
    """Save detailed results to CSV."""
    rows = []
    for result in results:
        qid = result["qid"]
        query = result["query"]
        expanded = result.get("expanded_query", query)
        
        # Get top 10 retrieved docs
        retrieved = result["retrieved"][:10]
        for rank, doc in enumerate(retrieved, 1):
            rows.append({
                "qid": qid,
                "query_en": queries_english.get(qid, "") if queries_english else "",
                "query_id": query,
                "expanded_query": expanded,
                "rank": rank,
                "doc_id": doc["id"],
                "score": doc["score"]
            })
    
    df = pd.DataFrame(rows)
    df.to_csv(output_path, index=False)
    logger.info(f"Detailed results saved to {output_path}")

## 8. Run Experiment Core


In [22]:
def run_embedding_qe(queries: Dict[str, str], corpus: List[Dict[str, str]], top_k: int = 10):
    expander = CrossLingualEmbeddingExpander()
    retriever = DenseRetriever()
    
    results = []
    # Using small subset for demonstration if needed, otherwise tqdm handles full iterator
    for qid, query in tqdm(list(queries.items()), desc="QE Evaluation"):
        expansion = expander.expand(query, num_terms=5)
        retrieved = retriever.retrieve([expansion.expanded_query], corpus, top_k=top_k)
        results.append({
            "qid": qid, "query": query, "expanded_query": expansion.expanded_query,
            "expansion_terms": expansion.expansion_terms, "retrieved": retrieved[0]
        })
    return results

def run_baseline(queries: Dict[str, str], corpus: List[Dict[str, str]], top_k: int = 10):
    retriever = DenseRetriever()
    queries_list = list(queries.values())
    retrieved_all = retriever.retrieve(queries_list, corpus, top_k=top_k)
    
    results = []
    for (qid, query), retrieved in zip(queries.items(), retrieved_all):
        results.append({
            "qid": qid, "query": query, "expanded_query": query, "retrieved": retrieved
        })
    return results

def run_bm25(queries: Dict[str, str], corpus: List[Dict[str, str]], top_k: int = 10):
    retriever = BM25Retriever()
    retriever.fit(corpus)
    queries_list = list(queries.values())
    retrieved_all = retriever.retrieve(queries_list, top_k=top_k)
    
    results = []
    for (qid, query), retrieved in zip(queries.items(), retrieved_all):
        results.append({
            "qid": qid, "query": query, "retrieved": retrieved
        })
    return results

## 9. Execute Experiments and Compare


In [ ]:
corpus, queries_en, queries_id, qrels = load_cosqa_data()
logger.info(f"Loaded {len(queries_en)} queries and {len(corpus)} documents")

top_k_eval = 10
method = "embedding" # Choices: "baseline", "bm25", "embedding"
output_file = "cosqa_benchmark_results.json"

print(f"--- Running {method.upper()} on English Queries ---")
if method == "embedding":
    results_en = run_embedding_qe(queries_en, corpus, top_k=top_k_eval)
elif method == "bm25":
    results_en = run_bm25(queries_en, corpus, top_k=top_k_eval)
else:
    results_en = run_baseline(queries_en, corpus, top_k=top_k_eval)

metrics_en = evaluate_results(results_en, qrels, top_k_eval)

print(f"--- Running {method.upper()} on Indonesian Queries ---")
if method == "embedding":
    results_id = run_embedding_qe(queries_id, corpus, top_k=top_k_eval)
elif method == "bm25":
    results_id = run_bm25(queries_id, corpus, top_k=top_k_eval)
else:
    results_id = run_baseline(queries_id, corpus, top_k=top_k_eval)
    
metrics_id = evaluate_results(results_id, qrels, top_k_eval)

print("\n" + "=" * 60)
print("BENCHMARK RESULTS COMPARISON")
print("=" * 60)
print(f"Method: {method} | Top-K: {top_k_eval}")
print("\n[English Queries]")
for k, v in metrics_en.items(): print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")
print("\n[Indonesian Queries]")
for k, v in metrics_id.items(): print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

ndcg_diff = metrics_id['NDCG@10'] - metrics_en['NDCG@10']
recall_diff = metrics_id['Recall@10'] - metrics_en['Recall@10']
print("\n[Difference (Indonesian - English)]")
print(f"NDCG@10: {ndcg_diff:+.4f}")
print(f"Recall@10: {recall_diff:+.4f}")

# Save JSON results
output_data = {
    "method": method,
    "top_k": top_k_eval,
    "metrics": {
        "english": metrics_en,
        "indonesian": metrics_id
    },
    "results": {
        "english": results_en,
        "indonesian": results_id
    }
}
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(output_data, f, indent=2, ensure_ascii=False)
logger.info(f"JSON metrics saved to {output_file}")

# Save CSV detailed results
csv_path_en = output_file.replace(".json", f"_{method}_english.csv")
csv_path_id = output_file.replace(".json", f"_{method}_indonesian.csv")

save_detailed_results(results_en, csv_path_en, queries_en)
save_detailed_results(results_id, csv_path_id, queries_en)
logger.info(f"Detailed CSV results saved to {csv_path_en} and {csv_path_id}")

README.md: 0.00B [00:00, ?B/s]

data/corpus-00000-of-00001-6ab5c5eda798a(…):   0%|          | 0.00/2.78M [00:00<?, ?B/s]

data/queries-00000-of-00001-4b7ee7cdfee8(…):   0%|          | 0.00/592k [00:00<?, ?B/s]

Generating corpus split:   0%|          | 0/20604 [00:00<?, ? examples/s]

Generating queries split:   0%|          | 0/20604 [00:00<?, ? examples/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


QE Evaluation:   0%|          | 0/20604 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/644 [00:00<?, ?it/s]